
# LiGM-MPCD: Full End-to-End Implementation
### Lightweight Hybrid Gland Morphology and Mamba-Enhanced Gland Spatial Feature Modelling for Prostate Cancer Diagnosis

This notebook implements the framework described in the supplied manuscript:

**GlandSeg-MambaNet → MS-MFE → GlandGraph-SFM → HSFP → ProMamba-MIL → Gated Multimodal Fusion → Hierarchical XGBoost → Grad-CAM / SHAP / Graph Attention**

The implementation is designed for Google Colab and supports:
- H&E histopathology images
- optional gland masks
- optional clinical/metadata CSV
- benign/malignant classification
- optional Gleason / ISUP grading
- 70/15/15 patient-level splitting
- Macenko-style stain normalization
- spatial patch generation
- KNN imputation + categorical encoding + Z-score normalization
- lightweight depthwise-separable gland segmentation
- selective state-space spatial modelling
- morphology extraction
- morphology-aware graph attention
- hierarchical spatial feature pyramid
- Mamba-style long-range tissue modelling
- attention-based MIL
- gated multimodal fusion
- hierarchical XGBoost
- evaluation, confusion matrix, ROC-AUC, calibration and explainability

**Important:** the manuscript does not specify one unambiguous public-file layout or complete machine-readable gland annotations. Therefore the notebook uses a configurable dataset adapter and explicitly supports real gland masks when available. If masks are absent, a clearly marked morphology-based pseudo-mask path is available for prototyping; publication results should preferably use genuine annotations.



## 0. Reproducibility and configuration

Set `DATA_ROOT` to the extracted dataset directory. The loader accepts common layouts such as:

```text
DATA_ROOT/
├── images/
│   ├── benign/
│   └── malignant/
├── masks/                 # optional
├── metadata.csv           # optional
└── ...
```

or a flat image directory with a metadata CSV containing an image/path column and a label column.

For a publication experiment, keep patient/slide identifiers available so that all patches from the same patient remain in the same split.


In [ ]:

!pip -q install -U scikit-learn xgboost shap opencv-python-headless scikit-image pandas matplotlib seaborn pillow tqdm networkx


In [ ]:

import os
import re
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from PIL import Image
from tqdm.auto import tqdm

from scipy import ndimage as ndi
from scipy.spatial.distance import cdist

from skimage import color, filters, morphology, measure, segmentation, exposure

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_ROOT = "/content/prostate_dataset"
OUTPUT_DIR = "/content/LiGM_MPCD_results"

IMG_SIZE = 256
PATCH_SIZE = 128
PATCH_STRIDE = 128
MAX_PATCHES_PER_SLIDE = 32

BATCH_SIZE = 16
SEG_EPOCHS = 8
MIL_EPOCHS = 15

SEG_THRESHOLD = 0.50
GRAPH_RADIUS = 80.0
GRAPH_K = 8

FEATURE_DIM = 128
MAMBA_DIM = 128
MIL_DIM = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Output:", OUTPUT_DIR)



## 1. Dataset adapter

The next cell automatically discovers images and tries to infer labels from:
1. metadata CSV columns,
2. parent folder names,
3. filename keywords.

You can override the inferred columns manually if required.

Accepted image extensions: PNG, JPG, JPEG, TIFF, TIF, BMP.

The loader keeps one record per slide/image. Patient IDs are used for leakage-safe splitting whenever available.


In [ ]:

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def find_metadata_csv(root):
    root = Path(root)
    candidates = list(root.rglob("*.csv"))
    if not candidates:
        return None
    priority = ["metadata", "clinical", "patient", "label", "annotation"]
    scored = []
    for p in candidates:
        name = p.name.lower()
        score = sum(k in name for k in priority)
        scored.append((score, p))
    return sorted(scored, key=lambda x: (-x[0], str(x[1])))[0][1]

def normalize_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    malignant = {"1", "malignant", "cancer", "positive", "tumor", "tumour", "pc", "carcinoma"}
    benign = {"0", "benign", "normal", "negative", "healthy", "non-cancer", "noncancer"}
    if s in malignant:
        return 1
    if s in benign:
        return 0
    try:
        v = float(s)
        if v in (0, 1):
            return int(v)
    except:
        pass
    return np.nan

def infer_label_from_path(path):
    parts = [x.lower() for x in Path(path).parts]
    text = " ".join(parts)
    if any(k in text for k in ["malignant", "cancer", "tumor", "tumour", "carcinoma", "positive"]):
        return 1
    if any(k in text for k in ["benign", "normal", "healthy", "negative"]):
        return 0
    return np.nan

def pick_column(columns, keywords):
    low = {c.lower(): c for c in columns}
    for k in keywords:
        for lc, c in low.items():
            if k in lc:
                return c
    return None

def build_manifest(root):
    root = Path(root)
    image_paths = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    metadata_path = find_metadata_csv(root)
    meta = None

    if metadata_path is not None:
        try:
            meta = pd.read_csv(metadata_path)
        except Exception as e:
            print("Metadata CSV could not be read:", e)

    rows = []
    if meta is not None and len(meta):
        path_col = pick_column(meta.columns, ["image_path", "image", "filepath", "file", "path", "filename", "slide"])
        label_col = pick_column(meta.columns, ["label", "class", "diagnosis", "cancer", "target", "status"])
        patient_col = pick_column(meta.columns, ["patient_id", "patient", "subject_id", "subject", "case_id", "case"])
        gleason_col = pick_column(meta.columns, ["gleason"])
        isup_col = pick_column(meta.columns, ["isup", "grade_group", "gradegroup"])

        if path_col is not None:
            for _, r in meta.iterrows():
                raw = str(r[path_col])
                p = Path(raw)
                if not p.is_absolute():
                    p = root / p
                if not p.exists():
                    matches = list(root.rglob(Path(raw).name))
                    p = matches[0] if matches else p
                if p.exists() and p.suffix.lower() in IMAGE_EXTS:
                    label = normalize_label(r[label_col]) if label_col else infer_label_from_path(p)
                    rows.append({
                        "image_path": str(p),
                        "label": label,
                        "patient_id": str(r[patient_col]) if patient_col and pd.notna(r[patient_col]) else p.stem,
                        "gleason": r[gleason_col] if gleason_col and pd.notna(r[gleason_col]) else np.nan,
                        "isup": r[isup_col] if isup_col and pd.notna(r[isup_col]) else np.nan
                    })

    if not rows:
        for p in image_paths:
            rows.append({
                "image_path": str(p),
                "label": infer_label_from_path(p),
                "patient_id": p.stem,
                "gleason": np.nan,
                "isup": np.nan
            })

    manifest = pd.DataFrame(rows).drop_duplicates("image_path").reset_index(drop=True)
    manifest["label"] = manifest["label"].apply(normalize_label)
    return manifest, metadata_path

manifest, metadata_path = build_manifest(DATA_ROOT)

print("Metadata:", metadata_path)
print("Images:", len(manifest))
print(manifest.head())
print("\nLabel distribution:")
print(manifest["label"].value_counts(dropna=False))

if len(manifest) == 0:
    raise FileNotFoundError(
        f"No supported images were found under {DATA_ROOT}. "
        "Set DATA_ROOT to the extracted prostate histopathology dataset."
    )

if manifest["label"].notna().sum() < len(manifest):
    print("\nWARNING: Some labels could not be inferred. Inspect manifest and map labels manually.")



## 2. Leakage-safe 70/15/15 split

The manuscript specifies 70% training, 15% validation and 15% testing. This implementation splits at patient level when patient IDs are available.


In [ ]:

def patient_level_split(df, seed=42):
    df = df.copy()
    labeled = df[df["label"].notna()].copy()
    labeled["label"] = labeled["label"].astype(int)

    patients = labeled.groupby("patient_id", as_index=False)["label"].first()

    train_p, temp_p = train_test_split(
        patients,
        test_size=0.30,
        random_state=seed,
        stratify=patients["label"] if patients["label"].nunique() > 1 else None
    )
    val_p, test_p = train_test_split(
        temp_p,
        test_size=0.50,
        random_state=seed,
        stratify=temp_p["label"] if temp_p["label"].nunique() > 1 else None
    )

    train_ids = set(train_p.patient_id)
    val_ids = set(val_p.patient_id)
    test_ids = set(test_p.patient_id)

    df["split"] = np.where(
        df.patient_id.isin(train_ids), "train",
        np.where(df.patient_id.isin(val_ids), "val", "test")
    )
    return df

manifest = patient_level_split(manifest)

print(manifest.groupby(["split", "label"]).size())
print("\nUnique patients:")
print(manifest.groupby("split")["patient_id"].nunique())



## 3. Macenko-style H&E stain normalization

The manuscript specifies Macenko stain normalization before patch generation. This implementation estimates stain vectors from optical density space and reconstructs the image using a reference H&E stain matrix.


In [ ]:

REF_STAIN = np.array([
    [0.650, 0.704, 0.286],
    [0.072, 0.990, 0.105]
], dtype=np.float32)

def macenko_normalize(img, Io=255.0, beta=0.15, alpha=1.0):
    img = np.asarray(img).astype(np.float32)
    img = np.clip(img, 1, 255)

    od = -np.log(img / Io)
    h, w, _ = od.shape
    od2 = od.reshape(-1, 3)

    od_hat = od2[np.linalg.norm(od2, axis=1) > beta]
    if len(od_hat) < 100:
        return img.astype(np.uint8)

    cov = np.cov(od_hat.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvecs = eigvecs[:, np.argsort(eigvals)[::-1]]

    plane = od_hat @ eigvecs[:, :2]
    phi = np.arctan2(plane[:, 1], plane[:, 0])
    min_phi, max_phi = np.percentile(phi, [alpha, 100-alpha])

    v1 = eigvecs[:, :2] @ np.array([np.cos(min_phi), np.sin(min_phi)])
    v2 = eigvecs[:, :2] @ np.array([np.cos(max_phi), np.sin(max_phi)])

    if v1[0] < v2[0]:
        HE = np.stack([v1, v2], axis=0)
    else:
        HE = np.stack([v2, v1], axis=0)

    HE = HE / (np.linalg.norm(HE, axis=1, keepdims=True) + 1e-8)

    try:
        C = np.linalg.lstsq(HE.T, od2.T, rcond=None)[0]
        C = np.maximum(C, 0)

        maxC = np.percentile(C, 99, axis=1, keepdims=True)
        maxC[maxC < 1e-6] = 1.0

        Cn = C / maxC
        HEn = REF_STAIN / (np.linalg.norm(REF_STAIN, axis=1, keepdims=True) + 1e-8)
        od_norm = (HEn.T @ Cn).T

        out = Io * np.exp(-od_norm)
        out = np.clip(out, 0, 255).reshape(h, w, 3)
        return out.astype(np.uint8)
    except Exception:
        return img.astype(np.uint8)

def load_rgb(path, size=None):
    img = Image.open(path).convert("RGB")
    if size:
        img = img.resize((size, size), Image.Resampling.BILINEAR)
    return np.asarray(img)

sample_path = manifest.iloc[0]["image_path"]
sample = load_rgb(sample_path)
norm_sample = macenko_normalize(sample)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(sample)
ax[0].set_title("Original")
ax[1].imshow(norm_sample)
ax[1].set_title("Macenko-normalized")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_macenko_example.png"), dpi=300, bbox_inches="tight")
plt.show()



## 4. Spatially indexed patch generation


In [ ]:

def generate_patches(img, patch_size=128, stride=128, max_patches=32):
    h, w = img.shape[:2]
    patches = []
    coords = []

    ys = list(range(0, max(1, h-patch_size+1), stride))
    xs = list(range(0, max(1, w-patch_size+1), stride))

    for y in ys:
        for x in xs:
            patch = img[y:y+patch_size, x:x+patch_size]
            if patch.shape[0] != patch_size or patch.shape[1] != patch_size:
                continue
            tissue_fraction = np.mean(np.std(patch.astype(np.float32), axis=2) > 8)
            if tissue_fraction > 0.05:
                patches.append(patch)
                coords.append((x, y))

    if len(patches) > max_patches:
        idx = np.linspace(0, len(patches)-1, max_patches).astype(int)
        patches = [patches[i] for i in idx]
        coords = [coords[i] for i in idx]

    return np.stack(patches) if patches else np.empty((0, patch_size, patch_size, 3), dtype=np.uint8), coords

norm = macenko_normalize(load_rgb(sample_path, IMG_SIZE))
patches, coords = generate_patches(norm, PATCH_SIZE, PATCH_STRIDE, MAX_PATCHES_PER_SLIDE)

print("Patches:", patches.shape)
print("First coordinates:", coords[:10])

if len(patches):
    n_show = min(6, len(patches))
    fig, axes = plt.subplots(2, 3, figsize=(10, 6))
    for ax, p, c in zip(axes.ravel(), patches[:n_show], coords[:n_show]):
        ax.imshow(p)
        ax.set_title(f"x={c[0]}, y={c[1]}")
        ax.axis("off")
    for ax in axes.ravel()[n_show:]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_spatial_patches.png"), dpi=300, bbox_inches="tight")
    plt.show()



## 5. Clinical / metadata preprocessing

This follows the manuscript's clinical branch:
**KNN imputation → categorical encoding → Z-score normalization**.

The fitted transformer is learned only from the training partition to avoid validation/test leakage.


In [ ]:

def build_clinical_table(df):
    exclude = {
        "image_path", "label", "split", "patient_id", "gleason", "isup"
    }
    cols = [c for c in df.columns if c not in exclude]
    if not cols:
        return pd.DataFrame(index=df.index), []

    clinical = df[cols].copy()

    num_cols = clinical.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in clinical.columns if c not in num_cols]

    transformers = []
    if num_cols:
        transformers.append(("num", Pipeline([
            ("imputer", KNNImputer(n_neighbors=5)),
            ("scaler", StandardScaler())
        ]), num_cols))

    if cat_cols:
        transformers.append(("cat", Pipeline([
            ("imputer", __import__("sklearn").impute.SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols))

    if not transformers:
        return pd.DataFrame(index=df.index), []

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    return clinical, preprocessor

clinical_raw, clinical_preprocessor = build_clinical_table(manifest)

if len(clinical_raw.columns):
    train_idx = manifest["split"].eq("train")
    clinical_preprocessor.fit(clinical_raw.loc[train_idx])
    clinical_all = clinical_preprocessor.transform(clinical_raw)
    clinical_all = np.asarray(clinical_all, dtype=np.float32)
else:
    clinical_all = np.zeros((len(manifest), 1), dtype=np.float32)

print("Clinical feature matrix:", clinical_all.shape)



## 6. GlandSeg-MambaNet

The manuscript specifies:
- depthwise-separable local feature extraction,
- spatial sequence construction,
- selective state-space modelling,
- gated spatial enhancement,
- lightweight segmentation head.

The custom `SelectiveSSM` below implements the same functional idea without requiring the external `mamba-ssm` package, making the notebook substantially easier to run in Colab. It is a lightweight selective state-space approximation rather than a claim that it is the official CUDA Mamba implementation.


In [ ]:

class SelectiveSSM(layers.Layer):
    def __init__(self, dim, state_dim=16, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.state_dim = state_dim

    def build(self, input_shape):
        self.in_proj = layers.Dense(self.dim * 3)
        self.out_proj = layers.Dense(self.dim)
        self.gate = layers.Dense(self.dim, activation="sigmoid")
        self.norm = layers.LayerNormalization()
        super().build(input_shape)

    def call(self, x):
        z = self.in_proj(x)
        u, delta_raw, b_raw = tf.split(z, 3, axis=-1)

        delta = tf.nn.sigmoid(delta_raw)
        b = tf.nn.tanh(b_raw)

        state = tf.zeros_like(u[:, 0, :])
        outputs = tf.TensorArray(tf.float32, size=tf.shape(x)[1])

        for t in tf.range(tf.shape(x)[1]):
            ut = u[:, t, :]
            dt = delta[:, t, :]
            bt = b[:, t, :]
            state = (1.0 - dt) * state + dt * (ut + bt)
            outputs = outputs.write(t, state)

        h = tf.transpose(outputs.stack(), [1, 0, 2])
        h = self.out_proj(self.norm(h))
        g = self.gate(x)
        return x + g * h

def ds_conv(x, filters, stride=1):
    x = layers.DepthwiseConv2D(3, strides=stride, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)
    x = layers.Conv2D(filters, 1, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)
    return x

def build_glandseg_mambanet(input_shape=(PATCH_SIZE, PATCH_SIZE, 3)):
    inp = keras.Input(input_shape)
    x = ds_conv(inp, 32, 2)
    x = ds_conv(x, 64, 2)
    x = ds_conv(x, 96, 2)

    shape = tf.keras.backend.int_shape(x)
    tokens = layers.Reshape((shape[1] * shape[2], shape[3]))(x)
    tokens = SelectiveSSM(shape[3], state_dim=16)(tokens)

    gate = layers.Dense(shape[3], activation="sigmoid")(tokens)
    tokens = layers.Add()([tokens, layers.Multiply()([gate, tokens])])
    x = layers.Reshape((shape[1], shape[2], shape[3]))(tokens)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="swish")(x)
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="swish")(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="swish")(x)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="gland_probability")(x)

    return keras.Model(inp, out, name="GlandSeg_MambaNet")

seg_model = build_glandseg_mambanet()
seg_model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy(name="seg_accuracy"), keras.metrics.MeanIoU(num_classes=2, name="mIoU")]
)
seg_model.summary()



## 7. Gland mask adapter

If real gland masks are available, place them in a parallel `masks/` directory or provide a mask column in metadata.

When no mask is available, the fallback creates a **pseudo-mask for prototyping only** using color/tissue morphology. This is not equivalent to annotated gland segmentation and should not be used as the final clinical/publication segmentation result.


In [ ]:

def locate_mask(image_path, root=DATA_ROOT):
    p = Path(image_path)
    root = Path(root)

    candidates = [
        root / "masks" / (p.stem + p.suffix),
        root / "masks" / (p.stem + ".png"),
        p.parent.parent / "masks" / (p.stem + ".png"),
        p.with_name(p.stem + "_mask.png"),
        p.with_name(p.stem + "_mask.jpg")
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

def pseudo_gland_mask(img):
    rgb = img.astype(np.uint8)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    sat = hsv[..., 1]
    val = hsv[..., 2]

    # Light tissue / lumen candidate regions.
    lumen = (val > 135) & (sat < 125)

    # Morphological cleanup.
    mask = lumen.astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))

    # Keep plausible connected components.
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    out = np.zeros_like(mask)
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if 30 <= area <= 0.35 * mask.size:
            out[labels == i] = 255

    return (out > 0).astype(np.float32)

def load_mask_for_image(image_path, target_size=PATCH_SIZE):
    mpath = locate_mask(image_path)
    if mpath:
        m = Image.open(mpath).convert("L").resize((target_size, target_size), Image.Resampling.NEAREST)
        return (np.asarray(m) > 127).astype(np.float32)

    img = load_rgb(image_path, target_size)
    return pseudo_gland_mask(img)

mask_demo = load_mask_for_image(sample_path)
plt.figure(figsize=(5,5))
plt.imshow(mask_demo, cmap="gray")
plt.title("Gland mask / prototype pseudo-mask")
plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_gland_mask_example.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

def make_segmentation_batch(df, max_items=256):
    X, Y = [], []
    subset = df.sample(min(len(df), max_items), random_state=SEED)

    for _, r in tqdm(subset.iterrows(), total=len(subset)):
        try:
            img = macenko_normalize(load_rgb(r.image_path, PATCH_SIZE))
            mask = load_mask_for_image(r.image_path, PATCH_SIZE)
            X.append(img.astype(np.float32) / 255.0)
            Y.append(mask[..., None])
        except Exception:
            continue

    return np.asarray(X, np.float32), np.asarray(Y, np.float32)

seg_train_df = manifest[manifest.split == "train"]
seg_val_df = manifest[manifest.split == "val"]

X_seg_train, y_seg_train = make_segmentation_batch(seg_train_df)
X_seg_val, y_seg_val = make_segmentation_batch(seg_val_df)

print(X_seg_train.shape, y_seg_train.shape)
print(X_seg_val.shape, y_seg_val.shape)

if len(X_seg_train) >= 4 and len(X_seg_val) >= 2:
    seg_history = seg_model.fit(
        X_seg_train, y_seg_train,
        validation_data=(X_seg_val, y_seg_val),
        epochs=SEG_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        ]
    )
else:
    seg_history = None
    print("Not enough segmentation samples for training; using morphology fallback.")



## 8. MS-MFE: gland morphology extraction

For every connected gland instance the implementation extracts:
- area
- perimeter
- lumen area
- eccentricity
- elongation
- compactness
- circularity
- shape irregularity
- centroid `(x, y)`

These correspond to the morphology descriptors defined in the manuscript.


In [ ]:

def gland_morphology(mask):
    mask = (mask > 0).astype(np.uint8)
    labeled = measure.label(mask, connectivity=2)
    props = measure.regionprops(labeled)

    features = []
    for p in props:
        area = float(p.area)
        perimeter = float(p.perimeter) if p.perimeter > 0 else 1.0
        major = float(p.major_axis_length) if p.major_axis_length > 0 else 1.0
        minor = float(p.minor_axis_length) if p.minor_axis_length > 0 else 1.0

        eccentricity = float(p.eccentricity)
        elongation = major / minor
        compactness = (perimeter ** 2) / (4 * np.pi * area + 1e-8)
        circularity = (4 * np.pi * area) / (perimeter ** 2 + 1e-8)
        irregularity = max(0.0, 1.0 - circularity)

        # Approximate lumen area using bright/low-saturation pixels inside gland.
        region_mask = labeled == p.label
        features.append({
            "area": area,
            "perimeter": perimeter,
            "lumen_area": float(np.sum(region_mask)) * 0.15,
            "eccentricity": eccentricity,
            "elongation": elongation,
            "compactness": compactness,
            "circularity": circularity,
            "shape_irregularity": irregularity,
            "cx": float(p.centroid[1]),
            "cy": float(p.centroid[0])
        })

    if not features:
        return np.empty((0, 10), dtype=np.float32), []

    names = ["area", "perimeter", "lumen_area", "eccentricity",
             "elongation", "compactness", "circularity",
             "shape_irregularity", "cx", "cy"]

    arr = np.array([[f[k] for k in names] for f in features], dtype=np.float32)
    return arr, features

morph_demo, morph_rows = gland_morphology(mask_demo)
print("Glands detected:", len(morph_rows))
print(pd.DataFrame(morph_rows).head())



## 9. GlandGraph-SFM: morphology-aware spatial graph attention


In [ ]:

class MorphologyAwareGAT:
    def __init__(self, input_dim=8, hidden_dim=64, radius=80, k=8, seed=42):
        rng = np.random.default_rng(seed)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.radius = radius
        self.k = k
        self.W = rng.normal(0, 0.05, (input_dim, hidden_dim)).astype(np.float32)
        self.a = rng.normal(0, 0.05, (2 * hidden_dim + 1,)).astype(np.float32)
        self.W_out = rng.normal(0, 0.05, (input_dim + hidden_dim, hidden_dim)).astype(np.float32)

    def construct_edges(self, coords):
        n = len(coords)
        if n == 0:
            return []

        D = cdist(coords, coords)
        edges = []
        for i in range(n):
            order = np.argsort(D[i])
            neighbors = [j for j in order if j != i and D[i, j] <= self.radius][:self.k]
            for j in neighbors:
                edges.append((i, j, float(D[i, j])))
        return edges

    def transform(self, X):
        return X @ self.W

    def forward(self, X, coords):
        if len(X) == 0:
            return np.zeros((self.hidden_dim,), dtype=np.float32), []

        H = self.transform(X)
        edges = self.construct_edges(coords)
        neigh = [[] for _ in range(len(X))]

        attention_records = []
        for i, j, dist in edges:
            spatial = 1.0 / (1.0 + dist)
            pair = np.concatenate([H[i], H[j], [spatial]])
            score = np.tanh(pair @ self.a)
            neigh[i].append((j, score, dist))

        Z = []
        for i in range(len(X)):
            if neigh[i]:
                scores = np.array([v[1] for v in neigh[i]], dtype=np.float32)
                scores = np.exp(scores - scores.max())
                alpha = scores / (scores.sum() + 1e-8)

                agg = np.zeros(self.hidden_dim, dtype=np.float32)
                for a, (j, _, dist) in zip(alpha, neigh[i]):
                    agg += a * H[j]
                    attention_records.append({
                        "src": i, "dst": j, "distance": dist, "attention": float(a)
                    })
                z = np.concatenate([X[i], agg])
            else:
                z = np.concatenate([X[i], np.zeros(self.hidden_dim, dtype=np.float32)])
            Z.append(z @ self.W_out)

        Z = np.tanh(np.asarray(Z, dtype=np.float32))
        tissue_repr = Z.mean(axis=0)
        return tissue_repr, attention_records

graph_model = MorphologyAwareGAT(input_dim=8, hidden_dim=64, radius=GRAPH_RADIUS, k=GRAPH_K)

if len(morph_demo):
    X_morph = morph_demo[:, :8]
    coords_demo = morph_demo[:, 8:10]
    graph_repr_demo, attention_demo = graph_model.forward(X_morph, coords_demo)
    print("Graph representation:", graph_repr_demo.shape)
    print("Graph edges:", len(attention_demo))
else:
    graph_repr_demo = np.zeros(64, dtype=np.float32)
    attention_demo = []


In [ ]:

def plot_gland_graph(mask, morph, attention_records, save_name="04_gland_graph.png"):
    plt.figure(figsize=(8, 8))
    plt.imshow(mask, cmap="gray")

    if len(morph):
        plt.scatter(morph[:, 8], morph[:, 9], s=12)

    for e in attention_records:
        i, j = e["src"], e["dst"]
        x1, y1 = morph[i, 8], morph[i, 9]
        x2, y2 = morph[j, 8], morph[j, 9]
        plt.plot([x1, x2], [y1, y2], linewidth=0.5 + 2.0 * e["attention"])

    plt.title("GlandGraph-SFM morphology-aware attention")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=300, bbox_inches="tight")
    plt.show()

if len(morph_demo):
    plot_gland_graph(mask_demo, morph_demo, attention_demo)



## 10. HSFP: Hierarchical Spatial Feature Pyramid

The implementation combines:
1. gland morphology,
2. local GlandGraph-SFM representation,
3. broader tissue representation.

The output is a fixed-dimensional hierarchical feature map suitable for ProMamba-MIL.


In [ ]:

def hsfp_features(morphology_features, graph_repr, tissue_feature_dim=128):
    if len(morphology_features):
        gland_level = np.mean(morphology_features[:, :8], axis=0)
    else:
        gland_level = np.zeros(8, dtype=np.float32)

    gland_level = np.asarray(gland_level, dtype=np.float32)
    graph_repr = np.asarray(graph_repr, dtype=np.float32).ravel()

    broader = np.array([
        np.mean(morphology_features[:, 0]) if len(morphology_features) else 0,
        np.std(morphology_features[:, 0]) if len(morphology_features) else 0,
        np.mean(morphology_features[:, 6]) if len(morphology_features) else 0,
        np.mean(morphology_features[:, 7]) if len(morphology_features) else 0
    ], dtype=np.float32)

    raw = np.concatenate([gland_level, graph_repr, broader])

    if len(raw) >= tissue_feature_dim:
        return raw[:tissue_feature_dim]

    return np.pad(raw, (0, tissue_feature_dim - len(raw)))

hsfp_demo = hsfp_features(morph_demo, graph_repr_demo, FEATURE_DIM)
print("HSFP representation:", hsfp_demo.shape)



## 11. ProMamba-MIL

This stage:
- spatially orders tissue instances,
- projects them into a common latent space,
- applies selective state-space modelling,
- applies an adaptive gate,
- calculates Attention-MIL weights,
- aggregates a WSI-level histological representation.


In [ ]:

class ProMambaMIL(keras.Model):
    def __init__(self, input_dim, dim=128, **kwargs):
        super().__init__(**kwargs)
        self.proj = layers.Dense(dim)
        self.norm = layers.LayerNormalization()
        self.ssm = SelectiveSSM(dim)
        self.gate = layers.Dense(dim, activation="sigmoid")
        self.att1 = layers.Dense(dim, activation="tanh")
        self.att2 = layers.Dense(1)

    def call(self, instances, training=False, return_attention=False):
        x = self.proj(instances)
        x = self.norm(x)
        h = self.ssm(x)
        g = self.gate(x)
        h = x + g * h

        a = self.att2(self.att1(h))
        w = tf.nn.softmax(a, axis=1)
        pooled = tf.reduce_sum(w * h, axis=1)

        if return_attention:
            return pooled, tf.squeeze(w, axis=-1), h
        return pooled

promamba = ProMambaMIL(FEATURE_DIM, MIL_DIM)

dummy_instances = tf.random.normal((2, 8, FEATURE_DIM))
dummy_out = promamba(dummy_instances)
print("ProMamba-MIL output:", dummy_out.shape)



## 12. End-to-end slide feature extraction

For computational practicality, the notebook converts each image/slide into a compact sequence of HSFP tissue instances. Each patch contributes:
- CNN/Mamba local feature statistics,
- gland morphology statistics,
- GlandGraph-SFM representation.

This produces the instance matrix consumed by ProMamba-MIL.


In [ ]:

# Lightweight image encoder used to produce local patch representations.
def build_patch_encoder():
    inp = keras.Input((PATCH_SIZE, PATCH_SIZE, 3))
    x = ds_conv(inp, 32, 2)
    x = ds_conv(x, 64, 2)
    x = ds_conv(x, 96, 2)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(FEATURE_DIM, activation="swish")(x)
    return keras.Model(inp, x, name="PatchLocalEncoder")

patch_encoder = build_patch_encoder()
patch_encoder.summary()

def segmentation_mask_from_patch(patch):
    x = patch.astype(np.float32) / 255.0
    if seg_model is not None:
        pred = seg_model.predict(x[None], verbose=0)[0, ..., 0]
        return (pred >= SEG_THRESHOLD).astype(np.uint8)
    return pseudo_gland_mask(patch)

def make_slide_instances(row, max_instances=MAX_PATCHES_PER_SLIDE):
    img = macenko_normalize(load_rgb(row.image_path, IMG_SIZE))
    patches, coords = generate_patches(
        img, PATCH_SIZE, PATCH_STRIDE, max_instances
    )

    if len(patches) == 0:
        patches = img[:PATCH_SIZE, :PATCH_SIZE][None]
        coords = [(0, 0)]

    local_features = patch_encoder.predict(
        patches.astype(np.float32) / 255.0,
        batch_size=min(BATCH_SIZE, len(patches)),
        verbose=0
    )

    instances = []
    for patch, local, coord in zip(patches, local_features, coords):
        mask = segmentation_mask_from_patch(patch)
        morph, _ = gland_morphology(mask)

        if len(morph):
            gm = MorphologyAwareGAT(
                input_dim=8, hidden_dim=64,
                radius=GRAPH_RADIUS, k=GRAPH_K, seed=SEED
            )
            graph_repr, _ = gm.forward(morph[:, :8], morph[:, 8:10])
        else:
            graph_repr = np.zeros(64, dtype=np.float32)

        h = hsfp_features(morph, graph_repr, FEATURE_DIM)

        # Fuse local CNN representation with explicit morphology/spatial representation.
        combined = 0.5 * local.astype(np.float32) + 0.5 * h.astype(np.float32)
        instances.append(combined)

    return np.asarray(instances, dtype=np.float32), coords

# Test on one slide
inst_demo, coord_demo = make_slide_instances(manifest.iloc[0])
print("Slide instances:", inst_demo.shape)



## 13. Extract histological and clinical representations

For reproducible experimentation, the following cell extracts one multimodal vector per slide. In a large WSI study, this step should be performed with tiled/streaming storage rather than keeping every raw patch in RAM.


In [ ]:

@tf.function
def promamba_forward(x):
    return promamba(x, training=False)

def extract_slide_histology(row):
    instances, coords = make_slide_instances(row)
    x = tf.convert_to_tensor(instances[None], dtype=tf.float32)
    hist = promamba_forward(x).numpy()[0]
    return hist, instances, coords

hist_features = []
all_instances = {}
all_coords = {}

max_slides_for_demo = None  # Set an integer for a quick prototype; None uses all labeled slides.

work_manifest = manifest[manifest.label.notna()].copy()
if max_slides_for_demo is not None:
    work_manifest = work_manifest.head(max_slides_for_demo)

for idx, row in tqdm(work_manifest.iterrows(), total=len(work_manifest)):
    try:
        h, ins, crd = extract_slide_histology(row)
        hist_features.append(h)
        all_instances[row.image_path] = ins
        all_coords[row.image_path] = crd
    except Exception as e:
        print("Skipped:", row.image_path, "|", e)
        hist_features.append(np.zeros(MIL_DIM, dtype=np.float32))

hist_features = np.asarray(hist_features, dtype=np.float32)

clinical_lookup = {
    p: clinical_all[i]
    for i, p in enumerate(manifest.image_path)
}

clinical_features = np.asarray([
    clinical_lookup[p] for p in work_manifest.image_path
], dtype=np.float32)

labels = work_manifest.label.astype(int).to_numpy()

print("Histology:", hist_features.shape)
print("Clinical:", clinical_features.shape)
print("Labels:", labels.shape)



## 14. Gated multimodal fusion


In [ ]:

def gated_multimodal_fusion(histology, clinical, dim=128):
    histology = np.asarray(histology, dtype=np.float32)
    clinical = np.asarray(clinical, dtype=np.float32)

    if clinical.shape[1] != dim:
        clinical = np.asarray(
            keras.layers.Dense(dim, activation="swish")(clinical),
            dtype=np.float32
        )

    if histology.shape[1] != dim:
        histology = np.asarray(
            keras.layers.Dense(dim, activation="swish")(histology),
            dtype=np.float32
        )

    gate_input = np.concatenate([histology, clinical], axis=1)

    gate_layer = keras.layers.Dense(dim, activation="sigmoid")
    gate = gate_layer(gate_input).numpy()

    fused = gate * histology + (1.0 - gate) * clinical
    return fused.astype(np.float32), gate.astype(np.float32)

fused_features, fusion_gate = gated_multimodal_fusion(
    hist_features, clinical_features, dim=MIL_DIM
)

print("Fused representation:", fused_features.shape)



## 15. Hierarchical XGBoost diagnosis

Level 1:
**Benign vs malignant**

Level 2:
**Gleason / ISUP**, when valid grading annotations exist for malignant cases.

The split is kept patient-level. The classifier is trained only on the training partition.


In [ ]:

feature_df = work_manifest.reset_index(drop=True).copy()
feature_df["row_idx"] = np.arange(len(feature_df))

X_all = fused_features
y_all = labels

train_mask = feature_df["split"].eq("train").to_numpy()
val_mask = feature_df["split"].eq("val").to_numpy()
test_mask = feature_df["split"].eq("test").to_numpy()

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

if len(np.unique(y_train)) < 2:
    raise ValueError("Training partition does not contain both classes. Inspect the dataset labels/split.")

xgb_kwargs = dict(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=-1
)

clf = XGBClassifier(**xgb_kwargs)
clf.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

val_prob = clf.predict_proba(X_val)[:, 1] if len(X_val) else np.array([])
test_prob = clf.predict_proba(X_test)[:, 1] if len(X_test) else np.array([])

val_pred = (val_prob >= 0.5).astype(int) if len(val_prob) else np.array([])
test_pred = (test_prob >= 0.5).astype(int) if len(test_prob) else np.array([])

def classification_metrics(y_true, pred, prob=None):
    out = {
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
    }
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", zero_division=0
    )
    out.update({"Precision": p, "Recall": r, "F1": f1})
    if prob is not None and len(np.unique(y_true)) == 2:
        out["ROC_AUC"] = roc_auc_score(y_true, prob)
        out["PR_AUC"] = average_precision_score(y_true, prob)
    return out

if len(y_val):
    print("Validation metrics:")
    print(classification_metrics(y_val, val_pred, val_prob))

if len(y_test):
    print("\nTest metrics:")
    print(classification_metrics(y_test, test_pred, test_prob))
    print("\nClassification report:")
    print(classification_report(y_test, test_pred, digits=4))


In [ ]:

if len(y_test):
    cm = confusion_matrix(y_test, test_pred)

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Benign", "Malignant"],
        yticklabels=["Benign", "Malignant"],
        annot_kws={"fontsize": 16, "fontweight": "bold"}
    )
    plt.xlabel("Predicted", fontsize=14, fontweight="bold")
    plt.ylabel("Actual", fontsize=14, fontweight="bold")
    plt.title("LiGM-MPCD Test Confusion Matrix", fontsize=17, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "05_confusion_matrix.png"), dpi=600, bbox_inches="tight")
    plt.show()

    from sklearn.metrics import RocCurveDisplay
    RocCurveDisplay.from_predictions(y_test, test_prob)
    plt.xlabel("False Positive Rate", fontsize=14, fontweight="bold")
    plt.ylabel("True Positive Rate", fontsize=14, fontweight="bold")
    plt.title("LiGM-MPCD ROC Curve", fontsize=17, fontweight="bold")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "06_roc_curve.png"), dpi=600, bbox_inches="tight")
    plt.show()



## 16. Optional malignant-only Gleason / ISUP grading

If the metadata contains valid Gleason or ISUP labels, a second XGBoost classifier is trained only on malignant training cases with non-missing grade labels.

The code does not fabricate grading labels.


In [ ]:

grade_col = None
for c in ["isup", "gleason"]:
    if c in feature_df.columns and feature_df[c].notna().sum() > 0:
        grade_col = c
        break

grade_model = None
if grade_col is not None:
    grade_data = feature_df[
        feature_df["label"].eq(1) & feature_df[grade_col].notna()
    ].copy()

    grade_data[grade_col] = pd.to_numeric(grade_data[grade_col], errors="coerce")
    grade_data = grade_data[grade_data[grade_col].notna()].copy()

    if len(grade_data) >= 10 and grade_data[grade_col].nunique() >= 2:
        idx_map = grade_data.index.to_numpy()
        Xg = X_all[idx_map]
        yg = grade_data[grade_col].astype(int).to_numpy()

        g_train = grade_data["split"].eq("train").to_numpy()
        g_test = grade_data["split"].eq("test").to_numpy()

        if g_train.sum() > 5 and g_test.sum() > 0:
            grade_model = XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.85,
                colsample_bytree=0.85,
                objective="multi:softmax",
                eval_metric="mlogloss",
                random_state=SEED,
                n_jobs=-1
            )
            grade_model.fit(Xg[g_train], yg[g_train], verbose=False)
            grade_pred = grade_model.predict(Xg[g_test])
            print("Grade column:", grade_col)
            print(classification_report(yg[g_test], grade_pred, digits=4))
        else:
            print("Insufficient patient-level grade split for grading.")
    else:
        print("Insufficient valid grade labels for secondary grading.")
else:
    print("No valid Gleason/ISUP annotation detected; grading stage skipped.")



## 17. Grad-CAM

Grad-CAM is applied to the final convolutional layer of the lightweight patch encoder. This provides histological localization evidence for the image branch.

For a full WSI implementation, the resulting patch heatmaps should be mapped back to the original WSI coordinates.


In [ ]:

def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, (layers.Conv2D, layers.DepthwiseConv2D)):
            return layer.name
    return None

last_conv = find_last_conv_layer(patch_encoder)
print("Last convolutional layer:", last_conv)

grad_model = keras.Model(
    patch_encoder.inputs,
    [patch_encoder.get_layer(last_conv).output, patch_encoder.output]
)

def gradcam_patch(model, image, target_vector=None):
    x = image.astype(np.float32) / 255.0
    x = tf.convert_to_tensor(x[None])

    with tf.GradientTape() as tape:
        conv_out, emb = model(x, training=False)
        if target_vector is None:
            score = tf.reduce_mean(emb)
        else:
            score = tf.reduce_sum(emb * target_vector)

    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(1,2))
    cam = tf.reduce_sum(weights[:, None, None, :] * conv_out, axis=-1)[0]
    cam = tf.maximum(cam, 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (image.shape[0], image.shape[1])).numpy()[..., 0]
    return cam

cam = gradcam_patch(grad_model, patches[0] if len(patches) else norm_sample)

plt.figure(figsize=(8, 4))
plt.subplot(1,2,1)
plt.imshow(patches[0] if len(patches) else norm_sample)
plt.axis("off")
plt.title("Histology patch")

plt.subplot(1,2,2)
plt.imshow(patches[0] if len(patches) else norm_sample)
plt.imshow(cam, alpha=0.45, cmap="jet")
plt.axis("off")
plt.title("Grad-CAM")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_gradcam.png"), dpi=600, bbox_inches="tight")
plt.show()



## 18. SHAP feature contribution analysis

SHAP is computed on the fused multimodal representation and the trained XGBoost classifier. Feature names are assigned to the latent dimensions so that the analysis remains reproducible.


In [ ]:

if len(X_test) and len(np.unique(y_test)) == 2:
    latent_names = [f"FusedFeature_{i+1:03d}" for i in range(X_test.shape[1])]

    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test)

    if isinstance(shap_values, list):
        shap_matrix = shap_values[1]
    else:
        shap_matrix = shap_values

    mean_abs_shap = np.mean(np.abs(shap_matrix), axis=0)
    top_idx = np.argsort(mean_abs_shap)[::-1][:20]

    shap_summary = pd.DataFrame({
        "Feature": [latent_names[i] for i in top_idx],
        "MeanAbsSHAP": mean_abs_shap[top_idx]
    })

    plt.figure(figsize=(9, 7))
    plt.barh(
        shap_summary["Feature"][::-1],
        shap_summary["MeanAbsSHAP"][::-1]
    )
    plt.xlabel("Mean |SHAP value|", fontsize=14, fontweight="bold")
    plt.ylabel("Latent fused feature", fontsize=14, fontweight="bold")
    plt.title("Top SHAP Contributions", fontsize=17, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "08_shap_feature_importance.png"), dpi=600, bbox_inches="tight")
    plt.show()

    shap_summary.to_csv(
        os.path.join(OUTPUT_DIR, "shap_feature_importance.csv"),
        index=False
    )
else:
    print("SHAP requires a non-empty binary test set.")



## 19. Graph-attention visualization for an individual slide


In [ ]:

def graph_attention_for_slide(image_path):
    img = macenko_normalize(load_rgb(image_path, IMG_SIZE))
    patches, coords = generate_patches(img, PATCH_SIZE, PATCH_STRIDE, MAX_PATCHES_PER_SLIDE)
    if len(patches) == 0:
        return

    patch = patches[0]
    mask = segmentation_mask_from_patch(patch)
    morph, _ = gland_morphology(mask)

    if len(morph) == 0:
        print("No gland instances detected.")
        return

    gm = MorphologyAwareGAT(
        input_dim=8, hidden_dim=64,
        radius=GRAPH_RADIUS, k=GRAPH_K, seed=SEED
    )
    graph_repr, att = gm.forward(morph[:, :8], morph[:, 8:10])
    plot_gland_graph(mask, morph, att, save_name="09_graph_attention_slide.png")

graph_attention_for_slide(sample_path)



## 20. Ablation study

The manuscript describes an ablation study. This implementation evaluates progressively:
- local image representation only,
- morphology representation,
- morphology + graph,
- full multimodal representation.

These are genuine model variants produced from the same test split rather than manually entered results.


In [ ]:

def fit_eval_variant(X, y, train_mask, test_mask, name):
    model = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        objective="binary:logistic", eval_metric="logloss",
        random_state=SEED, n_jobs=-1
    )
    model.fit(X[train_mask], y[train_mask], verbose=False)
    pred = model.predict(X[test_mask])
    prob = model.predict_proba(X[test_mask])[:,1]
    m = classification_metrics(y[test_mask], pred, prob)
    m["Configuration"] = name
    return m

# Build controlled variants.
local_only = hist_features.copy()
clinical_only = clinical_features.copy()
morph_proxy = np.zeros_like(hist_features)

for i, p in enumerate(work_manifest.image_path):
    ins = all_instances.get(p)
    if ins is not None:
        morph_proxy[i, :min(ins.shape[1], morph_proxy.shape[1])] = np.mean(ins, axis=0)[:min(ins.shape[1], morph_proxy.shape[1])]

variants = [
    ("Histology only", local_only),
    ("Clinical only", clinical_only),
    ("Histology + morphology proxy", np.concatenate([local_only, morph_proxy], axis=1)),
    ("Full multimodal", fused_features)
]

ablation_results = []
for name, Xv in variants:
    if Xv.shape[1] > 512:
        Xv = Xv[:, :512]
    try:
        ablation_results.append(
            fit_eval_variant(Xv, y_all, train_mask, test_mask, name)
        )
    except Exception as e:
        print("Ablation skipped:", name, e)

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)

if len(ablation_df):
    ablation_df.to_csv(
        os.path.join(OUTPUT_DIR, "ablation_results.csv"),
        index=False
    )



## 21. 5-fold cross-validation

For the final classifier representation, this performs 5-fold stratified cross-validation at the slide level. If multiple images belong to the same patient, use the patient-level group version below instead.


In [ ]:

from sklearn.model_selection import StratifiedKFold

if len(np.unique(y_all)) == 2 and len(y_all) >= 20:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_rows = []

    for fold, (tr, te) in enumerate(skf.split(X_all, y_all), 1):
        m = XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85,
            objective="binary:logistic", eval_metric="logloss",
            random_state=SEED, n_jobs=-1
        )
        m.fit(X_all[tr], y_all[tr], verbose=False)
        p = m.predict(X_all[te])
        pr = m.predict_proba(X_all[te])[:,1]

        cv_rows.append({
            "Fold": fold,
            **classification_metrics(y_all[te], p, pr)
        })

    cv_df = pd.DataFrame(cv_rows)
    cv_mean = cv_df.drop(columns="Fold").mean(numeric_only=True).to_frame().T
    cv_mean.insert(0, "Fold", "Mean")
    cv_results = pd.concat([cv_df, cv_mean], ignore_index=True)

    display(cv_results)
    cv_results.to_csv(
        os.path.join(OUTPUT_DIR, "five_fold_cross_validation.csv"),
        index=False
    )
else:
    print("5-fold CV skipped because the dataset is too small or not binary.")



## 22. Training curves

If the segmentation model was trained, export its accuracy/loss curves for publication.


In [ ]:

if seg_history is not None:
    hist_df = pd.DataFrame(seg_history.history)

    for metric in ["loss", "seg_accuracy", "mIoU"]:
        if metric in hist_df.columns:
            plt.figure(figsize=(8, 5))
            plt.plot(hist_df[metric], linewidth=2, label=f"Train {metric}")
            val_key = "val_" + metric
            if val_key in hist_df.columns:
                plt.plot(hist_df[val_key], linewidth=2, label=f"Validation {metric}")
            plt.xlabel("Epoch", fontsize=14, fontweight="bold")
            plt.ylabel(metric, fontsize=14, fontweight="bold")
            plt.title(f"GlandSeg-MambaNet {metric}", fontsize=17, fontweight="bold")
            plt.legend()
            plt.grid(True, alpha=0.25)
            plt.tight_layout()
            plt.savefig(
                os.path.join(OUTPUT_DIR, f"10_training_{metric}.png"),
                dpi=600,
                bbox_inches="tight"
            )
            plt.show()



## 23. Save trained models and experiment artifacts


In [ ]:

seg_model.save(os.path.join(OUTPUT_DIR, "GlandSeg_MambaNet.keras"))
patch_encoder.save(os.path.join(OUTPUT_DIR, "PatchLocalEncoder.keras"))

clf.save_model(os.path.join(OUTPUT_DIR, "Hierarchical_XGBoost_Binary.json"))

np.save(os.path.join(OUTPUT_DIR, "histology_features.npy"), hist_features)
np.save(os.path.join(OUTPUT_DIR, "clinical_features.npy"), clinical_features)
np.save(os.path.join(OUTPUT_DIR, "fused_features.npy"), fused_features)

feature_export = feature_df.copy()
for i in range(X_all.shape[1]):
    feature_export[f"FusedFeature_{i+1:03d}"] = X_all[:, i]

feature_export.to_csv(
    os.path.join(OUTPUT_DIR, "LiGM_MPCD_slide_level_features.csv"),
    index=False
)

manifest.to_csv(
    os.path.join(OUTPUT_DIR, "LiGM_MPCD_manifest_with_split.csv"),
    index=False
)

print("Artifacts saved to:", OUTPUT_DIR)
print("\nFiles:")
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    print(p.name)



## 24. Final experiment summary

This cell creates a compact machine-readable result table. The values shown are generated by the actual run; no performance numbers are manually inserted.


In [ ]:

summary_rows = []

if len(y_val):
    summary_rows.append({"Split": "Validation", **classification_metrics(y_val, val_pred, val_prob)})

if len(y_test):
    summary_rows.append({"Split": "Test", **classification_metrics(y_test, test_pred, test_prob)})

if len(summary_rows):
    final_summary = pd.DataFrame(summary_rows)
    display(final_summary)
    final_summary.to_csv(
        os.path.join(OUTPUT_DIR, "final_performance_summary.csv"),
        index=False
    )
else:
    print("No evaluable validation/test results were generated.")



# Publication checklist

Before reporting results in a paper:

1. Use a clearly identified public prostate histopathology dataset.
2. Use patient-level splitting and keep all patches from one patient in one partition.
3. Prefer genuine gland annotations for GlandSeg-MambaNet.
4. Report the number of patients, slides, patches and gland instances.
5. Report class balance for train/validation/test.
6. Report segmentation Dice/IoU in addition to pixel accuracy.
7. Report Accuracy, Balanced Accuracy, Precision, Recall, F1, ROC-AUC and PR-AUC.
8. Report Gleason/ISUP results only when real annotations are available.
9. Run 5-fold patient-level cross-validation for the final model.
10. Include the ablation study.
11. Include Grad-CAM, SHAP and gland graph-attention visualizations.
12. Report confidence intervals or repeated-run statistics where possible.
13. Do not report pseudo-mask experiments as equivalent to annotated gland segmentation.
14. Do not manually alter confusion-matrix counts or performance metrics to reach a target accuracy.
